In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Load data
df = pd.read_csv('train.csv')

# 2. Fix corrupted columns
for col in ['humidity', 'wind_speed', 'pressure']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Fix outliers BEFORE imputing missing values
df['irradiance'] = df['irradiance'].clip(lower=0)
df['cloud_coverage'] = df['cloud_coverage'].clip(upper=100)
df['temperature'] = df['temperature'].clip(upper=90)
df['voltage'] = df['voltage'].clip(upper=100)

# 4. Treat zero-efficiency as missing (confirmed data artifact, not real failures)
df.loc[df['efficiency'] == 0, 'efficiency'] = pd.NA
df['efficiency'] = pd.to_numeric(df['efficiency'], errors='coerce')

# 5. Fill missing categorical values
for col in ['error_code', 'installation_type']:
    df[col] = df[col].fillna('missing')

# 6. Fill missing numeric values (median)
numeric_missing_cols = ['maintenance_count', 'panel_age', 'soiling_ratio',
                         'cloud_coverage', 'temperature', 'voltage', 'irradiance',
                         'module_temperature', 'current', 'pressure', 'humidity',
                         'wind_speed', 'efficiency']

for col in numeric_missing_cols:
    df[col] = df[col].fillna(df[col].median())

# 7. Confirm clean state
print("Total rows:", len(df))
print("Total missing values:", df.isnull().sum().sum())
df.head()

Total rows: 20000
Total missing values: 0


,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,efficiency
0,0,7.817315,576.179270,41.243087,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,62.494044,12.824912,1018.866505,A1,missing,missing,0.562096
1,1,24.785727,240.003973,1.359648,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,43.851238,12.012044,1025.623854,D4,E00,dual-axis,0.396447
2,2,46.652695,687.612799,91.265368,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,49.704133,1.814400,1010.922654,C3,E00,missing,0.573776
3,3,53.339567,735.141179,96.190955,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,67.361473,8.736259,1021.846663,A1,missing,dual-axis,0.629009
4,4,5.575374,12.241203,27.495073,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,3.632000,0.522684,1008.555958,B2,E00,fixed,0.341874


In [2]:
# Check unique values in categorical columns before encoding
print("error_code values:")
print(df['error_code'].value_counts())

print("\ninstallation_type values:")
print(df['installation_type'].value_counts())

print("\nstring_id sample (checking if it's useful or just an ID):")
print(df['string_id'].head())
print("Unique string_id count:", df['string_id'].nunique())

error_code values:
error_code
E00        5977
missing    5912
E01        4100
E02        4011
Name: count, dtype: int64

installation_type values:
installation_type
tracking     5067
missing      5028
fixed        4990
dual-axis    4915
Name: count, dtype: int64

string_id sample (checking if it's useful or just an ID):
0    A1
1    D4
2    C3
3    A1
4    B2
Name: string_id, dtype: object
Unique string_id count: 4


In [3]:
# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=['error_code', 'installation_type', 'string_id'], drop_first=True)

print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

Shape before encoding: (20000, 17)
Shape after encoding: (20000, 23)


,id,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,...,efficiency,error_code_E01,error_code_E02,error_code_missing,installation_type_fixed,installation_type_missing,installation_type_tracking,string_id_B2,string_id_C3,string_id_D4
0,0,7.817315,576.179270,41.243087,32.135501,4.0,0.803199,37.403527,1.963787,13.691147,...,0.562096,False,False,True,False,True,False,False,False,False
1,1,24.785727,240.003973,1.359648,19.977460,8.0,0.479456,21.843315,0.241473,27.545096,...,0.396447,False,False,False,False,False,False,False,False,True
2,2,46.652695,687.612799,91.265368,1.496401,4.0,0.822398,48.222882,4.191800,43.363708,...,0.573776,False,False,False,False,True,False,False,True,False
3,3,53.339567,735.141179,96.190955,18.491582,3.0,0.837529,46.295748,0.960567,57.720436,...,0.629009,False,False,True,False,False,False,False,False,False
4,4,5.575374,12.241203,27.495073,30.722697,6.0,0.551833,0.000000,0.898062,6.786263,...,0.341874,False,False,False,True,False,False,True,False,False


In [4]:
# Drop id - not a real feature, just a row label
df_encoded = df_encoded.drop(columns=['id'])

# Save cleaned + encoded dataset for use outside this notebook
df_encoded.to_csv('train_cleaned.csv', index=False)

print("Final shape:", df_encoded.shape)
print("Saved as train_cleaned.csv")

Final shape: (20000, 22)
Saved as train_cleaned.csv


In [5]:
import os
print(os.getcwd())

C:\Users\Rohith Gowda


In [6]:
import os
print(os.getcwd())

C:\Users\Rohith Gowda
